# Steel defect examples
Put the extracted package in My Drive/Steel_External_Package. This is a defect-centred visual comparison. Use clean images for model input. Green overlays are source annotations or manual reference boxes. For unbiased detection scoring, use a fixed field of view and transform all ground-truth boxes.

In [ ]:
from google.colab import drive
from pathlib import Path
import sys
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Steel_External_Package')
sys.path.insert(0,str(ROOT))
from prepare_steel import prepare, save_manifest, gc10, severstal
OUT = ROOT/'output'
phone = OUT/'phone'
source = next((phone/'raw').glob('*.jpeg'))
save_manifest(phone,[prepare(source,phone,'07b',(158,515,355,908),'scratches','07b')])

## GC10-DET
Download an annotated COCO export from https://huggingface.co/datasets/imaadd05/gc10-det or its linked Roboflow source. Put a split's JSON at input/gc10/annotations.json and the corresponding images under input/gc10/images. The cell selects single-instance inclusion images using source annotations. Review output overlays before use.

In [ ]:
gc_root = ROOT/'input/gc10'
gc_labels = gc_root/'annotations.json'
if gc_labels.is_file():
    print('GC10 outputs:',len(gc10(gc_labels,gc_root/'images',OUT/'gc10')))
else:
    print(f'Place the GC10 COCO export at {gc_labels}')

## Severstal
Download the labeled training images and train.csv from https://www.kaggle.com/competitions/severstal-steel-defect-detection/data. Put them under input/severstal. Inspect samples in the source dataset and enter the image name and native ClassId in severstal_review.csv for each visibly relevant sample. Use rows with one compact defect region; the exported box encloses all mask pixels in that row, including any disconnected regions. Native ClassIds are preserved because the source does not establish their equivalence to NEU-DET. Review these as class-agnostic examples.

In [ ]:
sev_root = ROOT/'input/severstal'
sev_labels = sev_root/'train.csv'
if sev_labels.is_file():
    print('Severstal outputs:',len(severstal(sev_labels,sev_root/'train_images',ROOT/'severstal_review.csv',OUT/'severstal')))
else:
    print(f'Place the Severstal annotations at {sev_labels}')

In [ ]:
from IPython.display import display
from PIL import Image
for dataset in ('phone','gc10','severstal'):
    samples = sorted((OUT/dataset/'annotated').glob('*.png'))
    print(dataset,len(samples))
    for sample in samples[:3]:
        display(Image.open(sample))